#### 1. Combine csv files into one dataframe

In [ ]:
import glob
import pandas as pandas
from sklearn.model_selection import train_test_split

csv_folder = "../data/CICIOT2023/*.csv"
file_paths = glob.glob(csv_folder)

print(f"Found {len(file_paths)} files")

dataframe_list = [pandas.read_csv(file) for file in file_paths]
dataframe_combined = pandas.concat(dataframe_list, ignore_index=True)
dataframe_combined = dataframe_combined.reset_index(drop=True)

#### 2. Drop rows with null values

In [ ]:
dataframe_combined.dropna()

In [ ]:
dataframe_combined.shape

In [ ]:
dataframe_combined['label'].value_counts()

#### 3. Create smaller dataframe

In [ ]:
def sample_or_take_all(group):
    sample_size = min(len(group), 1000)
    return group.sample(n=sample_size, random_state=42)

dataframe_sampled = dataframe_combined.groupby('label').apply(sample_or_take_all)
dataframe_sampled = dataframe_sampled.reset_index(level=0)

In [ ]:
dataframe_sampled['label'].value_counts()

#### 4. Separate target from features

In [ ]:
import numpy as numpy

targets = dataframe_sampled['label']
features = dataframe_sampled.select_dtypes(include=[numpy.number])

In [ ]:
features.shape

#### 5. K Fold validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

model = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', random_state=0, gamma='scale', C=1.0)
)

k_fold = KFold(n_splits=5, shuffle=True, random_state=0)

cv_results = cross_val_score(model, features, targets, cv=k_fold, scoring='accuracy', n_jobs=-1)

print(cv_results)
cv_results.mean()

#### 5b. Stratified K Fold Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

strat_model = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', random_state=0, gamma='scale', C=10.0, class_weight='balanced')
)

strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scoring_metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

strat_results = cross_validate(
    strat_model,
    features,
    targets,
    scoring=scoring_metrics,
    cv=strat_kfold,
    n_jobs=-1
)

print(strat_results)

#### A. Split training and testing

In [ ]:
from sklearn.model_selection import train_test_split

features_train, features_test, targets_train, targets_test = train_test_split(features, targets, test_size=0.2, random_state=42)

#### B. Apply Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_test_scaled = scaler.transform(features_test)

#### C. Train Model

In [ ]:
from sklearn.svm import SVC

svc = SVC(kernel='rbf', random_state=0, gamma=1, C=1)
model2 = svc.fit(features_train_scaled, targets_train)